### Pipeline summary

| Step | Substep | What it does |
|---|---|---|
| Setup | — | Import libraries and set paths, models, and thresholds. |
| 1. Transcript | 1.1 | Read every mp4 in `NEWS_FOLDERS`. |
| 1. Transcript | 1.2 | Write a 16 kHz mono WAV under `outputs/audio`. Skip a file that is already there. |
| 1. Transcript | 1.3 | Transcribe each WAV with Parakeet. Keep word timestamps. |
| 1. Transcript | 1.4 | Build sentences. A sentence ends on `?`, `!`, or a final period. `U.S.` stays inside the sentence. |
| 1. Transcript | 1.5 | Save words and sentences under `outputs/transcripts`. |
| 2. Shots | 2.1 | Mark a cut where the scene score is above 0.3. |
| 2. Shots | 2.2 | Save the cut times under `outputs/shots`. Reuse the file on a later run. |
| 2. Shots | 2.3 | Move a sentence edge onto a cut within 0.4 seconds only when the cut sits in the gap beside the sentence. |
| 2. Shots | 2.4 | Run that snap for every video. |
| 3. Clips | 3.1 | Read the snapped sentences. |
| 3. Clips | 3.2 | From each sentence, grow a clip up to 30 seconds and another up to 45 seconds. |
| 3. Clips | 3.3 | Drop a clip shorter than 12 seconds. |
| 3. Clips | 3.4 | Save the candidates under `outputs/clips`. |
| 4. Scoring | 4.1 | Load the clips and the transcript sentences for each video. |
| 4. Scoring | 4.2 | Score centrality, lead, density, and clarity from 0 to 1. Lead uses 25% of that video’s length. Density uses the classifier plus MMR. |
| 4. Scoring | 4.3 | Mix the four scores with weights 0.35, 0.20, 0.15, and 0.30. Write `importance` back onto each clip. |
| 5. Stories | 5.1 | Drop a clip whose importance is below 0.5. |
| 5. Stories | 5.2 | Write one JSON per clip under `outputs/stories`. The filename ends with the score. |
| 5. Stories | 5.3 | Cut the mp4 into `outputs/stories/clips/thirty_sec_clips` and `forty_five_sec_clips`. |
| 5. Stories | 5.4 | Inside each length folder, drop a lower-scoring clip that overlaps a kept clip by more than 0.2. |
| 5. Stories | 5.5 | Keep the top 3 clips per video in each length folder. |


### Setup — Imports
Libraries for audio, transcription, scoring, and progress.


In [1]:
import json
import math
import re
import subprocess
import wave
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoModelForTDT, AutoProcessor, AutoTokenizer, pipeline

/home/rakesh/Downloads/ShortTok/5.VideoStories/VideoStories/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1.1 — Choose the videos
Paths, models, and thresholds. `NEWS_FOLDERS` is the list of video folders.


In [2]:
NEWS_FOLDERS = ["data/middle_east_war", "data/brics"]
AUDIO_DIR = Path("outputs/audio")
TRANSCRIPT_DIR = Path("outputs/transcripts")
SHOT_DIR = Path("outputs/shots")
CLIP_DIR = Path("outputs/clips")
STORIES_DIR = Path("outputs/stories")
STORY_CLIP_DIR = Path("outputs/stories/clips")

MODELS = Path("models")
TRANSCRIPTION_MODEL_ID = "nvidia/parakeet-tdt-0.6b-v3"
EMBED_ID = "sentence-transformers/all-MiniLM-L6-v2"
CLASSIFICATION_ID = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"

SCENE_THRESHOLD = 0.3
SNAP_SECONDS = 0.4
CLIP_LIMITS = (30, 45)
MIN_CLIP_SECONDS = 12
OVERLAP = 0.2
MIN_IMPORTANCE = 0.5
TOP_CLIPS = 3
LEAD_FRACTION = 0.25
CENTRALITY_WEIGHT = 0.35
LEAD_WEIGHT = 0.20
DENSITY_WEIGHT = 0.15
CLARITY_WEIGHT = 0.30
MMR_LAMBDA = 0.7
FACT_LABELS = (
    "a self-contained news fact naming who, what, a number, or a place that makes complete sense on its own",
    "filler, a greeting, or a line that refers to another clip",
)

### Step 1.2 — Extract audio
Write a 16 kHz mono WAV. Skip a file that is already there.


In [3]:
def extract_wav(video: Path) -> Path:
    wav = AUDIO_DIR / video.parent.name / f"{video.stem}.wav"
    wav.parent.mkdir(parents=True, exist_ok=True)
    if wav.exists():
        return wav
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(video), "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(wav)],
        check=True,
        capture_output=True,
    )
    return wav

### Step 1.2 — Run audio extraction
Extract a WAV for every video in `NEWS_FOLDERS` and print its duration.


In [4]:
for folder in NEWS_FOLDERS:
    for video in tqdm(sorted(Path(folder).glob("*.mp4")), desc="audio"):
        wav = extract_wav(video)
        with wave.open(str(wav)) as handle:
            seconds = handle.getnframes() / handle.getframerate()
        tqdm.write(f"{seconds:6.1f}s  {wav.name}")

audio: 100%|██████████| 5/5 [00:00<00:00, 715.24it/s]


 338.0s  1.Iran threatens US bases as Trump claims Hormuz oil shipments exceed pre-war levels_Al Jazeera English_720p.wav
 258.0s  2.All indicators point to conflict in the Middle East in the coming weeks: Jonathan Panikoff_CNBC Television_720p.wav
 327.1s  3.War in the Middle East: Israel launches West Bank operations_FRANCE 24 English_720p.wav
 770.3s  4.Iran threatens US bases as US and Israel reportedly consider new strikes_DW News_720p.wav
 113.0s  5.Oil Climbs on Threats of a Widening War in Middle East_Bloomberg Television_720p.wav


audio:  25%|██▌       | 1/4 [00:00<00:00,  4.96it/s]

 295.9s  1.BRICS: What is in the joint statement from the 11 members?_FRANCE 24 English_720p.wav
 131.7s  2.Day one of BRICS summit in New Delhi- members show support for Palestine_Al Jazeera English_720p.wav


audio:  75%|███████▌  | 3/4 [00:00<00:00,  4.36it/s]

 710.1s  3.BRICS Summit 2026: Vladimir Putin Accuses West Of Blocking Waterways, Seizing Ships | Russia | N18G_CNBC-TV18_720p.wav


audio: 100%|██████████| 4/4 [00:00<00:00,  4.05it/s]

 592.4s  4.Modi Seeks BRICS Unity as Global Tensions Rise_Bloomberg Television_720p.wav


### Step 1.3–1.4 — Transcribe
Load Parakeet and turn each WAV into words and sentences. A sentence ends on `?`, `!`, or a final period. `U.S.` stays inside the sentence.


In [5]:
processor = AutoProcessor.from_pretrained(TRANSCRIPTION_MODEL_ID, cache_dir=MODELS)
model = AutoModelForTDT.from_pretrained(TRANSCRIPTION_MODEL_ID, cache_dir=MODELS, dtype=torch.float16).cuda().eval()


def transcribe(wav):
    with wave.open(str(wav)) as handle:
        audio = np.frombuffer(handle.readframes(handle.getnframes()), dtype=np.int16).astype("float32") / 32768
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt").to("cuda")
    inputs["input_features"] = inputs["input_features"].to(torch.float16)
    with torch.inference_mode():
        output = model.generate(**inputs, return_dict_in_generate=True, max_new_tokens=4096)
    _, tokens = processor.decode(output.sequences, durations=output.durations, skip_special_tokens=True)

    words, text, start, end = [], "", None, None
    for token in tokens[0]:
        piece = token["token"]
        if text and piece[:1].isspace():
            words.append({"word": text, "start": start, "end": end})
            text, start = "", None
        if start is None:
            start = token["start"]
        text += piece.lstrip(" ")
        end = token["end"]
    if text:
        words.append({"word": text, "start": start, "end": end})

    groups, group = [], []
    for word in words:
        group.append(word)
        token = word["word"]
        if token.endswith(("?", "!")) or (token.endswith(".") and "." not in token[:-1]):
            groups.append(group)
            group = []
    if group:
        groups.append(group)
    sentences = [
        {"text": " ".join(word["word"] for word in group), "start": group[0]["start"], "end": group[-1]["end"]}
        for group in groups
    ]
    return words, sentences


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3/resolve/main/chat_template.json
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3/resolve/main/chat_template.json
Retrying in 2s [Retry 2/5].
Loading weights: 100%|██████████| 723/723 [00:01<00:00, 430.24it/s]


### Step 1.5 — Save transcripts
Transcribe every WAV and write words and sentences to `outputs/transcripts`.


In [6]:
transcripts = {}
for folder in NEWS_FOLDERS:
    wavs = sorted((AUDIO_DIR / Path(folder).name).glob("*.wav"))
    for wav in tqdm(wavs, desc="transcripts"):
        words, sentences = transcribe(wav)
        transcripts[wav] = {"words": words, "sentences": sentences}
        out = TRANSCRIPT_DIR / wav.parent.name / f"{wav.stem}.json"
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(transcripts[wav], indent=2))
        tqdm.write(f"{len(sentences):3} sentences  {out.name}")
        tqdm.write(sentences[0]["text"])


transcripts:  20%|██        | 1/5 [00:07<00:29,  7.27s/it]

 34 sentences  1.Iran threatens US bases as Trump claims Hormuz oil shipments exceed pre-war levels_Al Jazeera English_720p.json
Iran's revolutionary guard is threatening to attack U.S. bases across the Middle East if American warships keep escorting ships through the Strait of Hormuz.


transcripts:  40%|████      | 2/5 [00:12<00:19,  6.34s/it]

 43 sentences  2.All indicators point to conflict in the Middle East in the coming weeks: Jonathan Panikoff_CNBC Television_720p.json
For more, let's bring in Jonathan Panakoff.


transcripts:  60%|██████    | 3/5 [00:19<00:13,  6.50s/it]

...

 13 sentences  2.Day one of BRICS summit in New Delhi- members show support for Palestine_Al Jazeera English_720p.json
Despite deep divisions inside the BRICS family, the Delhi Declaration has seen consensus on several issues.


transcripts:  75%|███████▌  | 3/4 [00:23<00:08,  8.90s/it]

 65 sentences  3.BRICS Summit 2026: Vladimir Putin Accuses West Of Blocking Waterways, Seizing Ships | Russia | N18G_CNBC-TV18_720p.json
Thank you very much.


transcripts: 100%|██████████| 4/4 [00:39<00:00,  9.81s/it]

 76 sentences  4.Modi Seeks BRICS Unity as Global Tensions Rise_Bloomberg Television_720p.json
Dan, this BRICS summit is really interesting because it came from kind of an off-handed remark that somebody made on the sidelines of Anga, and now it has become a real thing and a real meeting.


### Step 2.1–2.3 — Find cuts and snap edges
Save a cut where the scene score is above 0.3. Move a sentence edge onto a cut only when that cut is within 0.4 seconds and in the gap beside the sentence.


In [7]:
def shot_cuts(video: Path) -> list[float]:
    out = SHOT_DIR / video.parent.name / f"{video.stem}.json"
    if out.exists():
        return json.loads(out.read_text())["cuts"]
    proc = subprocess.run(
        [
            "ffmpeg", "-hide_banner", "-nostats", "-i", str(video),
            "-filter:v", f"select='gt(scene,{SCENE_THRESHOLD})',metadata=print",
            "-an", "-f", "null", "-",
        ],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr)
    cuts = [float(line.rsplit("pts_time:", 1)[1]) for line in proc.stderr.splitlines() if "pts_time:" in line]
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({"cuts": cuts}))
    return cuts


def snap_sentences(sentences, cuts):
    snapped, moves = [], []
    for i, sentence in enumerate(sentences):
        prev_end = sentences[i - 1]["end"] if i else 0.0
        next_start = sentences[i + 1]["start"] if i + 1 < len(sentences) else sentence["end"] + SNAP_SECONDS
        start_cut = _nearest_cut(sentence["start"], cuts, prev_end, sentence["start"])
        end_cut = _nearest_cut(sentence["end"], cuts, sentence["end"], next_start)
        start = start_cut if start_cut is not None else sentence["start"]
        end = end_cut if end_cut is not None else sentence["end"]
        if start >= end:
            start, end, start_cut, end_cut = sentence["start"], sentence["end"], None, None
        snapped.append({**sentence, "start": start, "end": end})
        if start_cut is not None and start != sentence["start"]:
            moves.append({"edge": "start", "old": sentence["start"], "new": start, "text": sentence["text"]})
        if end_cut is not None and end != sentence["end"]:
            moves.append({"edge": "end", "old": sentence["end"], "new": end, "text": sentence["text"]})
    return snapped, moves


def _nearest_cut(time, cuts, low, high):
    choices = [cut for cut in cuts if low <= cut <= high and abs(cut - time) <= SNAP_SECONDS]
    return min(choices, key=lambda cut: abs(cut - time)) if choices else None

### Step 2.2–2.4 — Run shot detection
Load or detect cuts for every video and snap the sentence edges.


In [8]:
snapped = {}
for folder in NEWS_FOLDERS:
    videos = sorted(Path(folder).glob("*.mp4"))
    for index, video in enumerate(tqdm(videos, desc="shots")):
        cuts = shot_cuts(video)
        transcript = json.loads((TRANSCRIPT_DIR / video.parent.name / f"{video.stem}.json").read_text())
        sentences, moves = snap_sentences(transcript["sentences"], cuts)
        snapped[video] = sentences
        tqdm.write(f"{len(cuts):3} cuts  {len(moves):3} snaps  {video.name[:48]}")

shots: 100%|██████████| 5/5 [00:00<00:00, 419.99it/s]


  8 cuts    3 snaps  1.Iran threatens US bases as Trump claims Hormuz
  1 cuts    0 snaps  2.All indicators point to conflict in the Middle
 33 cuts    2 snaps  3.War in the Middle East: Israel launches West B
 38 cuts    3 snaps  4.Iran threatens US bases as US and Israel repor
  2 cuts    0 snaps  5.Oil Climbs on Threats of a Widening War in Mid


shots:  25%|██▌       | 1/4 [00:01<00:04,  1.53s/it]

 26 cuts    3 snaps  1.BRICS: What is in the joint statement from the


shots:  50%|█████     | 2/4 [00:02<00:02,  1.13s/it]

 19 cuts    5 snaps  2.Day one of BRICS summit in New Delhi- members 


shots:  75%|███████▌  | 3/4 [00:06<00:02,  2.57s/it]

 73 cuts    2 snaps  3.BRICS Summit 2026: Vladimir Putin Accuses West


shots: 100%|██████████| 4/4 [00:10<00:00,  2.56s/it]

 24 cuts    1 snaps  4.Modi Seeks BRICS Unity as Global Tensions Rise


### Step 3.2–3.3 — Build clip candidates
Grow 30-second and 45-second clips from whole sentences. Drop a clip shorter than 12 seconds.


In [9]:
def clip_candidates(sentences):
    clips = []
    for limit in CLIP_LIMITS:
        for i, sentence in enumerate(sentences):
            chosen = []
            for nxt in sentences[i:]:
                if nxt["end"] - sentence["start"] > limit:
                    break
                chosen.append(nxt)
            if not chosen:
                continue
            start, end = chosen[0]["start"], chosen[-1]["end"]
            if end - start < MIN_CLIP_SECONDS:
                continue
            clips.append({
                "start": start,
                "end": end,
                "duration": end - start,
                "max_len": limit,
                "text": " ".join(item["text"] for item in chosen),
            })
    return clips

### Step 3.1 and 3.4 — Save clips
Read the snapped sentences and write the candidates to `outputs/clips`.


In [10]:
clips = {}
for folder in NEWS_FOLDERS:
    for index, video in enumerate(tqdm(sorted(Path(folder).glob("*.mp4")), desc="clips")):
        transcript = json.loads((TRANSCRIPT_DIR / video.parent.name / f"{video.stem}.json").read_text())
        sentences, _ = snap_sentences(transcript["sentences"], shot_cuts(video))
        found = clip_candidates(sentences)
        clips[video] = found
        out = CLIP_DIR / video.parent.name / f"{video.stem}.json"
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps({"clips": found}, indent=2))
        counts = {limit: sum(clip["max_len"] == limit for clip in found) for limit in CLIP_LIMITS}
        tqdm.write(f"{counts[30]:3} x 30s  {counts[45]:3} x 45s  {video.name[:48]}")

clips: 100%|██████████| 5/5 [00:00<00:00, 348.51it/s]


 30 x 30s   30 x 45s  1.Iran threatens US bases as Trump claims Hormuz
 38 x 30s   38 x 45s  2.All indicators point to conflict in the Middle
 25 x 30s   30 x 45s  3.War in the Middle East: Israel launches West B
103 x 30s  103 x 45s  4.Iran threatens US bases as US and Israel repor
 16 x 30s   16 x 45s  5.Oil Climbs on Threats of a Widening War in Mid


clips: 100%|██████████| 4/4 [00:00<00:00, 357.47it/s]

 40 x 30s   40 x 45s  1.BRICS: What is in the joint statement from the
 11 x 30s   12 x 45s  2.Day one of BRICS summit in New Delhi- members 
 59 x 30s   63 x 45s  3.BRICS Summit 2026: Vladimir Putin Accuses West
 75 x 30s   75 x 45s  4.Modi Seeks BRICS Unity as Global Tensions Rise


### Step 4. Scoring — 4.2–4.3 Score clips
Load the embedding and classification models. Score centrality, lead, density, and clarity, then mix them into one importance score.


In [11]:
embed_model = SentenceTransformer(EMBED_ID, cache_folder=str(MODELS))
classification_model = pipeline(
    "zero-shot-classification",
    model=AutoModelForSequenceClassification.from_pretrained(CLASSIFICATION_ID, cache_dir=MODELS),
    tokenizer=AutoTokenizer.from_pretrained(CLASSIFICATION_ID, cache_dir=MODELS),
    device=0,
)
CLARITY_LABELS = ["standalone news naming who did what", "greeting or sign-off"]


def load_story(video):
    folder, stem = video.parent.name, video.stem
    clips = json.loads((CLIP_DIR / folder / f"{stem}.json").read_text())["clips"]
    sentences = json.loads((TRANSCRIPT_DIR / folder / f"{stem}.json").read_text())["sentences"]
    return clips, sentences


def _unit(values):
    lo, hi = min(values), max(values)
    if hi - lo < 1e-9:
        return [1.0] * len(values)
    return [(value - lo) / (hi - lo) for value in values]


def centrality(clips, sentences):
    video_vector = embed_model.encode([sentence["text"] for sentence in sentences], normalize_embeddings=True).mean(0)
    video_vector = video_vector / np.linalg.norm(video_vector)
    sims = embed_model.encode([clip["text"] for clip in clips], normalize_embeddings=True) @ video_vector
    return _unit(sims.tolist())


def density(clips):
    rates = []
    for clip in clips:
        parts = [part.strip() for part in re.split(r"(?<=[.?!])\s+", clip["text"]) if part.strip()]
        if not parts:
            rates.append(0.0)
            continue
        outs = classification_model(parts, list(FACT_LABELS), batch_size=8)
        if isinstance(outs, dict):
            outs = [outs]
        facts = [out["scores"][out["labels"].index(FACT_LABELS[0])] for out in outs]
        vecs = embed_model.encode(parts, normalize_embeddings=True)
        total, chosen = 0.0, []
        for i, fact in enumerate(facts):
            gain = fact
            if chosen:
                sim = max(float(vecs[i] @ vecs[j]) for j in chosen)
                gain = MMR_LAMBDA * fact - (1 - MMR_LAMBDA) * sim
            total += max(gain, 0.0)
            chosen.append(i)
        rates.append(total / clip["duration"])
    top = max(rates) or 1.0
    return [rate / top for rate in rates]


def clarity(clips):
    labels = CLARITY_LABELS
    outs = classification_model([clip["text"] for clip in clips], labels, batch_size=8)
    if isinstance(outs, dict):
        outs = [outs]
    return [out["scores"][out["labels"].index(labels[0])] for out in outs]


def lead(clips, sentences):
    length = max(sentence["end"] for sentence in sentences)
    return [math.exp(-clip["start"] / (LEAD_FRACTION * length)) for clip in clips]


def score_clips(clips, sentences):
    parts = {
        "centrality": centrality(clips, sentences),
        "lead": lead(clips, sentences),
        "density": density(clips),
        "clarity": clarity(clips),
    }
    for index, clip in enumerate(clips):
        piece = {name: values[index] for name, values in parts.items()}
        total = (
            CENTRALITY_WEIGHT * piece["centrality"]
            + LEAD_WEIGHT * piece["lead"]
            + DENSITY_WEIGHT * piece["density"]
            + CLARITY_WEIGHT * piece["clarity"]
        )
        clip["parts"] = piece
        clip["importance"] = min(1.0, max(0.0, total))
    return clips

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15338.11it/s]
/home/rakesh/Downloads/ShortTok/5.VideoStories/VideoStories/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 15558.42it/s]


### Step 4. Scoring — 4.1 and 4.3 Save scores
Load each video’s clips and transcript, score the clips, and write `importance` back to `outputs/clips`.


In [12]:
scored = {}
for folder in NEWS_FOLDERS:
    videos = sorted(Path(folder).glob("*.mp4"))
    for video in tqdm(videos, desc="scores"):
        clips, sentences = load_story(video)
        score_clips(clips, sentences)
        out = CLIP_DIR / video.parent.name / f"{video.stem}.json"
        out.write_text(json.dumps({"clips": clips}, indent=2))
        scored[video] = clips
        best = max(clips, key=lambda clip: clip["importance"])
        tqdm.write(f"{best['importance']:.2f}  {video.name[:48]}")
        tqdm.write(f"  {best['text'][:90]}")

scores:  20%|██        | 1/5 [00:01<00:07,  1.77s/it]

0.84  1.Iran threatens US bases as Trump claims Hormuz
  Iran's revolutionary guard is threatening to attack U.S. bases across the Middle East if A


scores:  40%|████      | 2/5 [00:04<00:06,  2.06s/it]

0.77  2.All indicators point to conflict in the Middle
  And I think that's the case right now, frankly, with Iran, right? That's the exact point t


scores:  60%|██████    | 3/5 [00:05<00:03,  1.70s/it]

...

0.63  2.Day one of BRICS summit in New Delhi- members 
  India's Prime Minister Nuraindamodi made the announcement this afternoon on day one of the


scores:  75%|███████▌  | 3/4 [00:05<00:02,  2.02s/it]

0.66  3.BRICS Summit 2026: Vladimir Putin Accuses West
  This is the statistics. Against this uh difficult backdrop, Russia has been working active


scores: 100%|██████████| 4/4 [00:10<00:00,  2.62s/it]

0.77  4.Modi Seeks BRICS Unity as Global Tensions Rise
  Uh India, of course, trying to sort of straddle all these different camps. Um, and then on


### Step 5 — Save stories and videos
Write one JSON per clip. Cut the mp4, drop a clip that overlaps a higher-scoring one by more than the overlap limit, and keep the top clips in each length.


In [13]:
def save_stories(video, clips):
    out_dir = STORIES_DIR / video.parent.name
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob(video.stem + "_*.json"):
        old.unlink()
    saved = []
    for clip in clips:
        path = out_dir / f"{video.stem}_{clip['importance']:.5f}.json"
        path.write_text(json.dumps(clip, indent=2))
        saved.append(path)
    return saved


def _overlap(a, b):
    inter = min(a["end"], b["end"]) - max(a["start"], b["start"])
    if inter <= 0:
        return 0.0
    union = (a["end"] - a["start"]) + (b["end"] - b["start"]) - inter
    return inter / union


def suppress_overlaps(clips):
    kept = []
    for clip in sorted(clips, key=lambda item: item["importance"], reverse=True):
        if any(_overlap(clip, other) > OVERLAP for other in kept):
            continue
        kept.append(clip)
    return kept


def save_story_clips(video, clips):
    folders = {30: "thirty_sec_clips", 45: "forty_five_sec_clips"}
    saved = []
    for limit, name in folders.items():
        out_dir = STORY_CLIP_DIR / name / video.parent.name
        out_dir.mkdir(parents=True, exist_ok=True)
        for old in out_dir.glob(video.stem + "_*.mp4"):
            old.unlink()
        for clip in suppress_overlaps([item for item in clips if item["max_len"] == limit])[:TOP_CLIPS]:
            path = out_dir / f"{video.stem}_{clip['importance']:.5f}.mp4"
            subprocess.run(
                [
                    "ffmpeg", "-y", "-ss", f"{clip['start']:.3f}", "-to", f"{clip['end']:.3f}",
                    "-i", str(video), "-c", "copy", "-avoid_negative_ts", "make_zero", str(path),
                ],
                check=True,
                capture_output=True,
            )
            saved.append(path)
    return saved

### Step 5 — Run story export
Keep clips whose score is at least 0.5. Write the JSON stories and the 30-second and 45-second videos.


In [14]:
stories = {}
for folder in NEWS_FOLDERS:
    for video in tqdm(sorted(Path(folder).glob("*.mp4")), desc="stories"):
        clips = json.loads((CLIP_DIR / video.parent.name / f"{video.stem}.json").read_text())["clips"]
        clips = [clip for clip in clips if clip["importance"] >= MIN_IMPORTANCE]
        saved = save_stories(video, clips)
        story_clips = save_story_clips(video, clips)
        stories[video] = saved
        best = max(clips, key=lambda clip: clip["importance"])
        tqdm.write(f"{len(story_clips):3} videos  {best['importance']:.5f}  {video.name[:40]}")

stories:  20%|██        | 1/5 [00:00<00:00,  6.59it/s]

  6 videos  0.84384  1.Iran threatens US bases as Trump claim


stories:  40%|████      | 2/5 [00:00<00:00,  7.97it/s]

  6 videos  0.77392  2.All indicators point to conflict in th


stories:  60%|██████    | 3/5 [00:00<00:00,  7.98it/s]

...

  4 videos  0.62611  2.Day one of BRICS summit in New Delhi- 


stories:  75%|███████▌  | 3/4 [00:00<00:00,  8.21it/s]

  6 videos  0.66351  3.BRICS Summit 2026: Vladimir Putin Accu


stories: 100%|██████████| 4/4 [00:00<00:00,  7.97it/s]

  6 videos  0.76654  4.Modi Seeks BRICS Unity as Global Tensi
